## Process - CNAF export of 2026-10-06, AEEH - deduplication against LAMP
- Load the AEEH CSV written by `10_06_2026_clean_cnaf_aeeh.ipynb`
- Load an export of LAMP's `public.beneficiaires` table (flat columns)
- Drop the beneficiaries LAMP already holds, so they are not inserted a second time
- Output the remaining ones to CSV, same format as the input

A beneficiary is already in LAMP when these all match (see
`partners.LAMP_DEDUPLICATION_KEY_COLUMNS`): child nom, prenom, date de naissance (day only),
genre, allocataire nom and prenom. Text is normalised the way LAMP's `normalise_recherche`
does: no accents, upper case, apostrophes dropped, hyphens as spaces.

Read-only on both inputs: the only file written is `DB_CNAF_EXPORT_2026_AEEH_DEDUPLICATED`.

In [ ]:
import os
import csv
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# partners_lib imports utils.data_utils, which lives at the data/ root: make that root
# importable first, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

# partners_lib itself sits one level up, in partners/, next to the other partner folders.
partners_root = str(Path.cwd().parent)
if partners_root not in sys.path:
    sys.path.append(partners_root)

import partners_lib as partners

load_dotenv()

cnaf_aeeh_input_filepath = os.environ['DB_CNAF_EXPORT_2026_AEEH']
lamp_beneficiaires_filepath = os.environ['LAMP_BENEFICIAIRES_EXPORT_PATHFILE_2026']
base_output_filepath = os.environ['DB_CNAF_EXPORT_2026_AEEH_DEDUPLICATED']

In [ ]:
# Kept as plain strings so the output is written back exactly as it was read
df_cnaf_aeeh = pd.read_csv(
    cnaf_aeeh_input_filepath, sep=';', dtype=str, keep_default_na=False, encoding='utf-8')

print(f"{len(df_cnaf_aeeh)} AEEH beneficiary row(s) read from {cnaf_aeeh_input_filepath}")

In [ ]:
# Only the key columns are needed from LAMP
df_lamp = pd.read_csv(
    lamp_beneficiaires_filepath, sep=';', dtype=str,
    usecols=partners.LAMP_DEDUPLICATION_KEY_COLUMNS, encoding='utf-8')

print(f"{len(df_lamp)} LAMP beneficiary row(s) read from {lamp_beneficiaires_filepath}")

In [ ]:
df_new, df_already_in_lamp = partners.drop_beneficiaries_already_in_lamp(df_cnaf_aeeh, df_lamp)

print(f"{len(df_already_in_lamp)} beneficiary row(s) already in LAMP, removed")
print(f"{len(df_new)} beneficiary row(s) left to insert")

In [ ]:
# Sample of the removed rows, for a manual check
df_already_in_lamp[['nom', 'prenom', 'date_naissance', 'genre', 'allocataire']].head(20)

In [ ]:
# output to CSV file, same format as 10_06_2026_clean_cnaf_aeeh.ipynb
df_new.to_csv(base_output_filepath, sep=';', index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

print(f"{len(df_new)} aeeh written to {base_output_filepath}")

In [ ]:
# Recap of the deduplication against LAMP
print(f"{len(df_lamp)} bénéficiaires dans l'export LAMP")
print(f"{len(df_cnaf_aeeh)} lignes AEEH en entrée")
print(f"  - {len(df_already_in_lamp)} déjà présentes dans LAMP (même nom, prénom, date de naissance, genre, nom et prénom de l'allocataire)")
print(f"{len(df_new)} lignes écrites dans {base_output_filepath}")
assert len(df_cnaf_aeeh) - len(df_already_in_lamp) == len(df_new)

## ⏭ Next: generate_new_codes.ipynb
Run `../generate_new_codes.ipynb` with `SOURCE = 'CNAF_AEEH'`, which reads
`DB_CNAF_EXPORT_2026_AEEH_DEDUPLICATED`, to assign these beneficiaries their `id_psp`.